<a href="https://colab.research.google.com/github/ananthchavancs24-prog/AI/blob/main/Lab_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import collections

# ========================================== #
# 1. SYNTAX TREE FOR FIRST-ORDER LOGIC (FOL) #
# ========================================== #

class Expr:
    def __or__(self, other): return Or(self, other)
    def __and__(self, other): return And(self, other)
    def __invert__(self): return Not(self)
    def impl(self, other): return Implies(self, other)

class Const(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name

class Var(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name

class Pred(Expr):
    def __init__(self, name, *args):
        self.name = name
        self.args = args
    def __repr__(self):
        return f"{self.name}({', '.join(map(str, self.args))})"

class Not(Expr):
    def __init__(self, arg): self.arg = arg
    def __repr__(self): return f"¬{self.arg}"

class Or(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class And(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Implies(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, formula): self.var, self.formula = var, formula
    def __repr__(self): return f"∀{self.var}: {self.formula}"

class Exists(Expr):
    def __init__(self, var, formula): self.var, self.formula = var, formula
    def __repr__(self): return f"∃{self.var}: {self.formula}"

# ========================================== #
# 2. CLAUSE CLASS REPRESENTATION FOR CNF     #
# ========================================== #

class Clause:
    def __init__(self, literals):
        self.literals = tuple(sorted(literals))

    def __repr__(self):
        if not self.literals:
            return "[] (Contradiction)"
        return " ∨ ".join(f"{'' if lit[0] else '¬'}{lit[1]}({', '.join(lit[2:])})" for lit in self.literals)

    def __eq__(self, other):
        return self.literals == other.literals

    def __hash__(self):
        return hash(self.literals)

# ========================================== #
# 3. FOL TO CNF CONVERSION STAGES            #
# ========================================== #

def eliminate_implications(expr):
    if isinstance(expr, Implies): return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    if isinstance(expr, Not): return Not(eliminate_implications(expr.arg))
    if isinstance(expr, Or): return Or(eliminate_implications(expr.left), eliminate_implications(expr.right))
    if isinstance(expr, And): return And(eliminate_implications(expr.left), eliminate_implications(expr.right))
    if isinstance(expr, ForAll): return ForAll(expr.var, eliminate_implications(expr.formula))
    if isinstance(expr, Exists): return Exists(expr.var, eliminate_implications(expr.formula))
    return expr

def move_negations_inward(expr):
    if isinstance(expr, Not):
        arg = expr.arg
        if isinstance(arg, Not): return move_negations_inward(arg.arg)
        if isinstance(arg, And): return Or(move_negations_inward(Not(arg.left)), move_negations_inward(Not(arg.right)))
        if isinstance(arg, Or): return And(move_negations_inward(Not(arg.left)), move_negations_inward(Not(arg.right)))
        if isinstance(arg, ForAll): return Exists(arg.var, move_negations_inward(Not(arg.formula)))
        if isinstance(arg, Exists): return ForAll(arg.var, move_negations_inward(Not(arg.formula)))
        return expr
    if isinstance(expr, Or): return Or(move_negations_inward(expr.left), move_negations_inward(expr.right))
    if isinstance(expr, And): return And(move_negations_inward(expr.left), move_negations_inward(expr.right))
    if isinstance(expr, ForAll): return ForAll(expr.var, move_negations_inward(expr.formula))
    if isinstance(expr, Exists): return Exists(expr.var, move_negations_inward(expr.formula))
    return expr

var_counter = 0
def standardize_variables(expr, mapping=None):
    global var_counter
    if mapping is None: mapping = {}
    if isinstance(expr, Var): return Var(mapping.get(expr.name, expr.name))
    if isinstance(expr, Const): return expr
    if isinstance(expr, Pred): return Pred(expr.name, *(standardize_variables(arg, mapping) for arg in expr.args))
    if isinstance(expr, Not): return Not(standardize_variables(expr.arg, mapping))
    if isinstance(expr, Or): return Or(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    if isinstance(expr, And): return And(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    if isinstance(expr, (ForAll, Exists)):
        var_counter += 1
        new_var_name = f"{expr.var.name}_{var_counter}"
        new_mapping = mapping.copy()
        new_mapping[expr.var.name] = new_var_name
        cls = ForAll if isinstance(expr, ForAll) else Exists
        return cls(Var(new_var_name), standardize_variables(expr.formula, new_mapping))
    return expr

def skolemize(expr, universal_vars=None):
    if universal_vars is None: universal_vars = []
    if isinstance(expr, Exists):
        sk_name = f"sk_{expr.var.name}"
        replacement = Const(sk_name) if not universal_vars else Pred(sk_name, *universal_vars)
        def subst(e):
            if isinstance(e, Var) and e.name == expr.var.name: return replacement
            if isinstance(e, Pred): return Pred(e.name, *(subst(arg) for arg in e.args))
            if isinstance(e, Not): return Not(subst(e.arg))
            if isinstance(e, Or): return Or(subst(e.left), subst(e.right))
            if isinstance(e, And): return And(subst(e.left), subst(e.right))
            if isinstance(e, ForAll): return ForAll(e.var, subst(e.formula))
            if isinstance(e, Exists): return Exists(e.var, subst(e.formula))
            return e
        return skolemize(subst(expr.formula), universal_vars)
    if isinstance(expr, ForAll): return ForAll(expr.var, skolemize(expr.formula, universal_vars + [expr.var]))
    if isinstance(expr, Or): return Or(skolemize(expr.left, universal_vars), skolemize(expr.right, universal_vars))
    if isinstance(expr, And): return And(skolemize(expr.left, universal_vars), skolemize(expr.right, universal_vars))
    if isinstance(expr, Not): return Not(skolemize(expr.arg, universal_vars))
    return expr

def drop_universal_quantifiers(expr):
    if isinstance(expr, ForAll): return drop_universal_quantifiers(expr.formula)
    if isinstance(expr, Or): return Or(drop_universal_quantifiers(expr.left), drop_universal_quantifiers(expr.right))
    if isinstance(expr, And): return And(drop_universal_quantifiers(expr.left), drop_universal_quantifiers(expr.right))
    if isinstance(expr, Not): return Not(drop_universal_quantifiers(expr.arg))
    return expr

def distribute_or_over_and(expr):
    if isinstance(expr, Or):
        left, right = distribute_or_over_and(expr.left), distribute_or_over_and(expr.right)
        if isinstance(left, And): return And(distribute_or_over_and(Or(left.left, right)), distribute_or_over_and(Or(left.right, right)))
        if isinstance(right, And): return And(distribute_or_over_and(Or(left, right.left)), distribute_or_over_and(Or(left, right.right)))
        return Or(left, right)
    if isinstance(expr, And): return And(distribute_or_over_and(expr.left), distribute_or_over_and(expr.right))
    return expr

def get_clauses(expr):
    if isinstance(expr, And): return get_clauses(expr.left) + get_clauses(expr.right)
    def collect_literals(e):
        if isinstance(e, Or): return collect_literals(e.left) + collect_literals(e.right)
        return [(False, e.arg.name, *(str(a) for a in e.arg.args))] if isinstance(e, Not) else [(True, e.name, *(str(a) for a in e.args))]
    return [collect_literals(expr)]

def fol_to_clauses(formula):
    return get_clauses(distribute_or_over_and(drop_universal_quantifiers(skolemize(standardize_variables(move_negations_inward(eliminate_implications(formula)))))))

# ========================================== #
# 4. RESOLUTION PROVER & UNIFICATION          #
# ========================================== #

def unify_var(var, val, subst):
    if var in subst:
        return unify(subst[var], val, subst)
    elif val in subst:
        return unify(var, subst[val], subst)
    else:
        subst[var] = val
        return subst

def unify(x, y, subst=None):
    if subst is None: subst = {}
    if subst is False: return False
    if x == y: return subst
    if isinstance(x, str) and (x.islower() or '_' in x):
        return unify_var(x, y, subst)
    if isinstance(y, str) and (y.islower() or '_' in y):
        return unify_var(y, x, subst)
    if isinstance(x, (list, tuple)) and isinstance(y, (list, tuple)) and len(x) == len(y):
        for xi, yi in zip(x, y):
            subst = unify(xi, yi, subst)
        return subst
    return False

def apply_subst(literal, subst):
    sign, pred, *args = literal
    new_args = tuple(subst.get(arg, arg) if (isinstance(arg, str) and (arg.islower() or '_' in arg)) else arg for arg in args)
    new_args = tuple(subst.get(a, a) for a in new_args)
    return (sign, pred, *new_args)

def resolve(clause1, clause2):
    resolvents = []
    for l1 in clause1.literals:
        for l2 in clause2.literals:
            if l1[0] != l2[0] and l1[1] == l2[1]:
                subst = unify(l1[2:], l2[2:])
                if subst is not False:
                    new_lits = set()
                    for lit in clause1.literals:
                        if lit != l1:
                            new_lits.add(apply_subst(lit, subst))
                    for lit in clause2.literals:
                        if lit != l2:
                            new_lits.add(apply_subst(lit, subst))
                    resolvents.append((Clause(new_lits), subst))
    return resolvents

# ========================================== #
# 5. DEFINE KNOWLEDGE BASE & RUN RESOLUTION   #
# ========================================== #

x = Var('x')
y = Var('y')
John = Const('John')
Anil = Const('Anil')
Peanuts = Const('Peanuts')
Apple = Const('Apple')
vegetables = Const('vegetables')
Harry = Const('Harry')

# Raw First-Order Logic (FOL) input representation
fol_kb = [
    ForAll(x, Pred('food', x).impl(Pred('likes', John, x))),                              # a
    And(Pred('food', Apple), Pred('food', vegetables)),                                   # b
    ForAll(x, ForAll(y, (Pred('eats', x, y) & ~Pred('killed', x)).impl(Pred('food', y)))),# c
    And(Pred('eats', Anil, Peanuts), Pred('alive', Anil)),                                # d
    ForAll(x, Pred('eats', Anil, x).impl(Pred('eats', Harry, x))),                         # e
    ForAll(x, (~Pred('killed', x)).impl(Pred('alive', x))),                               # f
    ForAll(x, Pred('alive', x).impl(~Pred('killed', x)))                                  # g
]

# Programmatically convert raw FOL to Clause structures
all_kb_clauses = []
for formula in fol_kb:
    clauses_lists = fol_to_clauses(formula)
    for cl_list in clauses_lists:
        norm_list = []
        for lit in cl_list:
            norm_args = [arg.lower() if '_' in arg else arg for arg in lit[2:]]
            norm_list.append((lit[0], lit[1], *norm_args))
        all_kb_clauses.append(Clause(norm_list))

# Define negated query programmatically
negated_query_clause = Clause([(False, 'likes', 'John', 'Peanuts')])

print("FOL converted to CNF Clauses:")
for idx, cl in enumerate(all_kb_clauses, 1):
    print(f"{idx}. {cl}")
print(f"Negated Query: {negated_query_clause}\n")

# Step-by-Step Resolution Refutation Proof Execution
current_clause = negated_query_clause
print("Resolution Proof Steps:")
print(f"Step 0 (Negated Query): {current_clause}")

path = [
    (all_kb_clauses[0], "Resolve with Clause 1: ¬food(x_1) ∨ likes(John, x_1)"),
    (all_kb_clauses[3], "Resolve with Clause 4: ¬eats(x_2, y_3) ∨ food(y_3) ∨ killed(x_2)"),
    (all_kb_clauses[4], "Resolve with Clause 5: eats(Anil, Peanuts)"),
    (all_kb_clauses[8], "Resolve with Clause 9: ¬alive(x_6) ∨ ¬killed(x_6)"),
    (all_kb_clauses[5], "Resolve with Clause 6: alive(Anil)")
]

for i, (kb_c, desc) in enumerate(path, 1):
    resolvents = resolve(current_clause, kb_c)
    if resolvents:
        next_clause, subst = resolvents[0]
        print(f"\n{desc}")
        print(f"Substitution used: {subst}")
        print(f"Step {i}: {next_clause}")
        current_clause = next_clause
    else:
        print(f"Failed to resolve at step {i}")
        break

if len(current_clause.literals) == 0:
    print("\nSuccess! Empty clause [] reached. The query 'likes(John, Peanuts)' is proven to be TRUE using Automated CNF Conversion & Resolution.")
else:
    print("\nProof search failed to find a contradiction.")


FOL converted to CNF Clauses:
1. ¬food(x_1) ∨ likes(John, x_1)
2. food(Apple)
3. food(vegetables)
4. ¬eats(x_2, y_3) ∨ food(y_3) ∨ killed(x_2)
5. eats(Anil, Peanuts)
6. alive(Anil)
7. ¬eats(Anil, x_4) ∨ eats(Harry, x_4)
8. alive(x_5) ∨ killed(x_5)
9. ¬alive(x_6) ∨ ¬killed(x_6)
Negated Query: ¬likes(John, Peanuts)

Resolution Proof Steps:
Step 0 (Negated Query): ¬likes(John, Peanuts)

Resolve with Clause 1: ¬food(x_1) ∨ likes(John, x_1)
Substitution used: {'x_1': 'Peanuts'}
Step 1: ¬food(Peanuts)

Resolve with Clause 4: ¬eats(x_2, y_3) ∨ food(y_3) ∨ killed(x_2)
Substitution used: {'y_3': 'Peanuts'}
Step 2: ¬eats(x_2, Peanuts) ∨ killed(x_2)

Resolve with Clause 5: eats(Anil, Peanuts)
Substitution used: {'x_2': 'Anil'}
Step 3: killed(Anil)

Resolve with Clause 9: ¬alive(x_6) ∨ ¬killed(x_6)
Substitution used: {'x_6': 'Anil'}
Step 4: ¬alive(Anil)

Resolve with Clause 6: alive(Anil)
Substitution used: {}
Step 5: [] (Contradiction)

Success! Empty clause [] reached. The query 'likes(John, Pea